In [100]:
import pandas as pd

In [101]:
url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

df = pd.read_csv(url)

In [102]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [103]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [104]:
# check missing values
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [105]:
cat_cols = list(df.select_dtypes(include="object").columns)
num_cols = list(df.select_dtypes(include="number").columns)

print(f"categorical features {cat_cols}")
print(f"numerical features {num_cols}")

categorical features ['lead_source', 'industry', 'employment_status', 'location']
numerical features ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']


In [106]:
# filling missing values
df[cat_cols] = df[cat_cols].fillna("missing")
df[num_cols] = df[num_cols].fillna(0)

In [107]:
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

#### Q 1 : What is the most frequent observation (mode) for the column industry?

In [108]:
df["industry"].mode()[0]

'technology'

#### Q2: Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count
- Only consider the pairs above when answering 

In [109]:
corr_matrix = df[num_cols].corr()
corr_matrix.loc[
    ["interaction_count", "number_of_courses_viewed", "annual_income"],
    ["lead_score", "interaction_count", "number_of_courses_viewed"]
]

,lead_score,interaction_count,number_of_courses_viewed
interaction_count,0.915746,1.000000,0.721609
number_of_courses_viewed,0.757204,0.721609,1.000000
annual_income,0.229496,0.122842,0.161300


#### Q 3: Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.Round the scores to 2 decimals using round(score, 2). 
Which of these variables has the biggest mutual information score?
- industry
- location
- lead_source
- employment_status

In [110]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [111]:
from sklearn.metrics import mutual_info_score
def calculate_mi(series):
    return round(mutual_info_score(df_train["converted"], series), 2)

In [112]:
mi = df_train[cat_cols].apply(calculate_mi)
print(mi.sort_values(ascending=False))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


##### Q4: 
 Now let's train a logistic regression. Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.

- Fit the model on the training dataset.
- To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
- model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

In [129]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [114]:
y_train = df_train["converted"]
y_val = df_val["converted"]

In [115]:
X_train_dict = df_train.drop(columns=["converted"]).to_dict(orient="records")
X_val_dict = df_val.drop(columns=["converted"]).to_dict(orient="records")

In [116]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(X_train_dict)
X_val = dv.transform(X_val_dict)

In [117]:
model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42 )
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [124]:
y_pred = model.predict(X_val)
original_accuracy = (y_pred == y_val).mean()
round(original_accuracy, 2)

np.float64(0.64)

#### Q6: 

In [130]:
col_to_test = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]
for col in col_to_test:
    train_dicts_reduced = [
        {k: v for k, v in row.items() if k != col}
        for row in X_train_dict
    ]

    val_dicts_reduced = [
        {k: v for k, v in row.items() if k != col}
        for row in X_val_dict
    ]

    dv_reduced = DictVectorizer(sparse=False)
    X_train_reduced = dv_reduced.fit_transform(train_dicts_reduced)
    X_val_reduced = dv_reduced.transform(val_dicts_reduced)
    
    model_reduced = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model_reduced.fit(X_train_reduced, y_train)
    
    y_pred = model_reduced.predict(X_val_reduced)
    accuracy = (y_pred == y_val).mean()

    difference = original_accuracy - accuracy

    print(
        f"Feature: {feature}, "
        f"Accuracy: {accuracy:.4f}, "
        f"Difference: {difference:.4f}"
    )

Feature: interaction_count, Accuracy: 0.6420, Difference: 0.0030
Feature: interaction_count, Accuracy: 0.6430, Difference: 0.0020
Feature: interaction_count, Accuracy: 0.6010, Difference: 0.0440


In [133]:
c = [0.000001, 0.00001, 0.0001, 0.001]

for i in c:
    model = LogisticRegression(solver='liblinear',C=i,
        max_iter=1000,random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    accuracy = (y_pred == y_val).mean()
    print(f"c = {i}, Accuracy = {accuracy:.3f}")

c = 1e-06, Accuracy = 0.598
c = 1e-05, Accuracy = 0.598
c = 0.0001, Accuracy = 0.613
c = 0.001, Accuracy = 0.645
